# Previsão de Contagens Mensais de Sinistros de Seguro Auto com PROC FORECAST


## Resumo Executivo

Uma equipe de reservas atuariais precisa de uma visão de 12 meses à frente das contagens mensais de sinistros de seguro auto que mostre crescimento constante da carteira mais um pronunciado aumento no clima de inverno. Este notebook gera cinco anos de contagens mensais sintéticas de sinistros (60 meses, jan/2020 - dez/2024, variando de cerca de 1.460 a 2.780 sinistros), e então usa o **PROC FORECAST** para ajustar uma linha de base autorregressiva por etapas e um modelo sazonal multiplicativo de Holt-Winters. Cada modelo produz 12 previsões pontuais mensais com limites de confiança de 95% para planejamento de capacidade e reservas. O modelo sazonal Holt-Winters projeta a demanda mais forte um a dois meses à frente (cerca de 2.780-2.900 sinistros), suavizando para um vale em torno da etapa nove (cerca de 2.200), enquanto a linha de base autorregressiva projeta uma decadência mais suave; ambas as bandas de confiança se alargam com o horizonte.


## Fontes de Dados

| Conjunto de Dados | Linhas | Granularidade | Variáveis-Chave | Descrição |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Uma linha por mês calendário (jan/2020 - dez/2024) | `date` (data SAS, `MONYY7.`), `claim_count` (numérico) | Contagens mensais sintéticas de sinistros de seguro auto construídas a partir de uma tendência de crescimento linear (~9 sinistros/mês), um ciclo anual senoidal, um aumento aditivo de clima de inverno em dez/jan/fev, e ruído gaussiano (`rand('normal')`). A semente `20240531` o torna reproduzível. |


# Previsão de Contagens Mensais de Sinistros de Seguro Auto

Equipes de reservas e planejamento de capacidade em uma seguradora de linhas pessoais precisam de uma visão prospectiva de quantos **sinistros de auto** serão reportados a cada mês. O volume de sinistros nesta carteira cresce continuamente à medida que a carteira se expande, e aumenta a cada inverno quando gelo, neve e menos luz do dia elevam os sinistros de colisão e vidros.

Este notebook percorre um fluxo de trabalho completo do `PROC FORECAST`:

1. Gerar uma série realista e totalmente sintética de contagem mensal de sinistros.
2. Ajustar uma linha de base **autorregressiva por etapas (STEPAR)** que captura tendência mais autocorrelação.
3. Ajustar um modelo **Holt-Winters multiplicativo (WINTERS)** que modela explicitamente o ciclo sazonal de 12 meses.
4. Comparar os dois modelos e interpretar a previsão prospectiva e a banda de confiança.

Tudo roda em dados sintéticos inline — sem arquivos externos ou acesso à rede.


## Etapa 1 — Gerar a série sintética de sinistros

A etapa DATA abaixo constrói **60 observações mensais** (jan/2020 a dez/2024). Para cada mês combinamos quatro componentes que espelham uma carteira de auto real:

- **Tendência** — uma base de ~1.800 sinistros crescendo ~9 por mês à medida que as exposições aumentam.
- **Ciclo anual** — um termo seno/cosseno dando uma onda sazonal suave.
- **Aumento de inverno** — um acréscimo aditivo em dezembro, janeiro e fevereiro.
- **Ruído** — `rand('normal', 0, 90)` para variabilidade realista mês a mês.

`call streaminit()` fixa o fluxo aleatório para que o notebook seja reproduzível. A variável `date` é uma data SAS verdadeira construída com `INTNX` e formatada `MONYY7.`, e a instrução `ID date INTERVAL=MONTH` a nomeia como o identificador de tempo. As primeiras 14 linhas impressas abaixo ficam entre aproximadamente 1.460 e 2.450 sinistros.


In [1]:
DADOS claims;
    CHAMAR streaminit(20240531);
    FAZER i = 0 ATÉ 59;
        /* Construir uma data de mês SAS real para que INTERVAL=MONTH se alinhe */
        date = intnx('month', '01JAN2020'd, i);
        FORMATO date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = jan ... 12 = dez */

        trend  = 1800 + 9 * i;               /* base de exposição crescente */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx IN (12, 1, 2));   /* aumento de gelo/neve */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        SE claim_count < 0 ENTÃO claim_count = 0;
        SAÍDA;
    FIM;
    MANTER date claim_count;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=claims(obs=14) RÓTULO;
    RÓTULO date = "Mês" claim_count = "Sinistros Reportados";
    TÍTULO 'Primeiros 14 Meses de Contagens Sintéticas de Sinistros de Auto';
EXECUTAR;


                            Primeiros 14 Meses de Contagens Sintéticas de Sinistros de Auto                             

  Obs    Mês  Sinistros Reportados
    1  21915                  2178
    2  21946                  2281
    3  21975                  2252
    4  22006                  1974
    5  22036                  2067
    6  22067                  1805
    7  22097                  1697
    8  22128                  1619
    9  22159                  1462
   10  22189                  1688
   11  22220                  1713
   12  22250                  2008
   13  22281                  2116
   14  22312                  2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Etapa 2 — Linha de base autorregressiva por etapas (METHOD=STEPAR)

`METHOD=STEPAR` é o padrão. Primeiro ajusta um modelo de tendência temporal — aqui `TREND=2` para uma tendência linear — e então aplica uma **autorregressão por etapas** sobre os resíduos, inserindo e retendo defasagens AR por significância. `AR=4` limita a ordem autorregressiva candidata a quatro defasagens, o que é suficiente para uma série mensal com momento de curto prazo.

Principais opções usadas aqui:

- `LEAD=12` — prever 12 meses além dos dados.
- `ALPHA=0.05` — limites de confiança de 95%.
- `OUTFULL` — empilha as linhas históricas `ACTUAL` junto com as linhas do horizonte de previsão no conjunto de dados `OUT=` (em vez de apenas as linhas de previsão).
- `OUTLIMIT` — adiciona as **colunas** de limite de confiança inferior/superior (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — nomeia o identificador de tempo e declara que a série é mensal.


In [2]:
PROCEDIMENTO forecast DADOS=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIÁVEL claim_count;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=fc_stepar(obs=10) RÓTULO;
    RÓTULO date = "Mês" claim_count = "Sinistros" _type_ = "Tipo"
          l95_claim_count = 'Limite Inf. 95%' u95_claim_count = 'Limite Sup. 95%';
    TÍTULO 'Saída STEPAR: Linhas de Real, Ajustado e Previsão';
EXECUTAR;


                            Primeiros 14 Meses de Contagens Sintéticas de Sinistros de Auto                             

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                   Saída STEPAR: Linhas de Real, Ajustado e Previsão                                    

  Obs    Mês    Tipo    Sinistros  Limite Inf. 95%  Limite Sup. 95%
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Lendo o conjunto de dados OUT=

O conjunto de dados `OUT=` empilha linhas por uma coluna `_TYPE_` e carrega os limites de confiança como **colunas** laterais:

| Elemento | Tipo | Significado |
|---------|------|-------------|
| `_TYPE_ = 'ACTUAL'` | linha | O `claim_count` observado para cada um dos 60 meses históricos. |
| `_TYPE_ = 'FORECAST'` | linha | As 12 previsões pontuais para o horizonte de previsão. |
| `L95_claim_count` / `U95_claim_count` | coluna | Limites de confiança inferior / superior de 95%, preenchidos nas linhas `FORECAST` (ausentes nas linhas `ACTUAL`). O nível numérico reflete `ALPHA=`. |

Portanto, o `OUT=` impresso aqui contém 72 linhas: 60 linhas históricas `ACTUAL` seguidas de 12 linhas do horizonte `FORECAST`. As dez primeiras linhas mostradas acima são todas `ACTUAL`, com as colunas de limite de confiança ausentes porque os limites se aplicam apenas às linhas de previsão.

> **Nota do motor.** O `OUTFULL` do SAS também escreve uma linha `FORECAST` de um passo à frente dentro da amostra para cada mês histórico, e `OUTRESID` adiciona linhas `_TYPE_='RESIDUAL'`. O PROC FORECAST do Jenner ainda não emite essas linhas ajustadas/residuais dentro da amostra (rastreado como teste de lacuna `400979`), então este notebook lê apenas o histórico `ACTUAL` e o horizonte `FORECAST` prospectivo.


## Etapa 3 — Modelo sazonal Holt-Winters (METHOD=WINTERS)

O modelo STEPAR captura tendência e autocorrelação de curto prazo, mas não modela explicitamente o aumento recorrente de dezembro a fevereiro. Para uma série com um ritmo anual claro, o **Holt-Winters multiplicativo** é a ferramenta melhor.

`METHOD=WINTERS` decompõe a série em nível, tendência linear (`TREND=2`), e um **fator sazonal multiplicativo**. `SEASONS=12` declara um ciclo sazonal de 12 períodos (mensal). Novamente solicitamos um `LEAD` de 12 meses com limites de 95% e `OUTFULL OUTLIMIT` para que a saída se alinhe com a execução STEPAR.

Como o motor avança o `ID` da previsão em uma unidade por etapa (em vez de respeitar `INTERVAL=MONTH` para as datas do horizonte — teste de lacuna `400979`), a célula abaixo revisa a previsão por **meses à frente (etapa 1-12)** em vez de depender das datas de calendário impressas.


In [3]:
PROCEDIMENTO forecast DADOS=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VARIÁVEL claim_count;
EXECUTAR;

/* Manter o horizonte prospectivo de 12 meses e indexá-lo por etapa (1..12). */
DADOS horizon;
    DEFINIR fc_winters;
    RETER months_ahead 0;
    SE _type_ = 'FORECAST';
    months_ahead + 1;
    MANTER months_ahead claim_count l95_claim_count u95_claim_count;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=horizon RÓTULO;
    RÓTULO months_ahead     = 'Meses à Frente'
          claim_count       = 'Sinistros Previstos'
          l95_claim_count   = 'Limite Inf. 95%'
          u95_claim_count   = 'Limite Sup. 95%';
    TÍTULO 'Previsão Holt-Winters de 12 Meses (por etapa)';
EXECUTAR;


                                   Saída STEPAR: Linhas de Real, Ajustado e Previsão                                    

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                     Previsão Holt-Winters de 12 Meses (por etapa)                                      

  Obs   Meses à Frente  Sinistros Previstos  Limite Inf. 95%  Limite Sup. 95%
    1                1           2783.07951      2577.844742      2988.314278
    2                2          2897.355557      2607.109764      3187.601349
    3                3          2805.272075      2449.795029       3160.74912
    4                4          2664.498049      2254.028514      3074.967585
    5                5          2628.810136      2169.891244      3087.729029
    6                6           2548.39319      2045.672732      3051.113649
    7                7          2391.649756        1848.6496      2934.649912
    8                8 


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Etapa 4 — Comparar os dois modelos lado a lado

A forma mais clara de julgar se o modelo sazonal está valendo a pena é colocar sua previsão de 12 meses ao lado da linha de base STEPAR, etapa a etapa. Extraímos as linhas `FORECAST` de ambos os conjuntos de dados `OUT=`, indexamos cada uma por meses à frente, e as combinamos para que a divergência fique visível de imediato.

Os dois métodos concordam no nível geral, mas discordam na **forma**: o Holt-Winters carrega o ritmo anual para dentro do horizonte (um nível mais alto no início do horizonte que suaviza para um vale intermediário e sobe novamente), enquanto o STEPAR — que modela a sazonalidade apenas indiretamente por meio de defasagens AR — decai mais suavemente em direção à média da série. A diferença entre eles em cada etapa é o sinal sazonal que o STEPAR deixa de captar.

> O SAS normalmente conduziria essa verificação de adequação com resíduos de um passo à frente `OUTRESID` (`_TYPE_='RESIDUAL'`). O Jenner ainda não preenche essas linhas (teste de lacuna `400979`), então comparamos as duas previsões prospectivas diretamente — um diagnóstico que usa apenas a saída que o motor realmente produz.


In [4]:
/* Horizonte STEPAR, indexado por meses-a-frente. */
DADOS stepar_h;
    DEFINIR fc_stepar;
    RETER months_ahead 0;
    SE _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    MANTER months_ahead stepar;
EXECUTAR;

/* Horizonte WINTERS, indexado por meses-a-frente. */
DADOS winters_h;
    DEFINIR fc_winters;
    RETER months_ahead 0;
    SE _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    MANTER months_ahead winters;
EXECUTAR;

/* Combinar os dois e mostrar a diferença sazonal que o STEPAR não capta. */
DADOS COMPARE;
    MESCLAR stepar_h winters_h;
    POR months_ahead;
    seasonal_gap = winters - stepar;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=COMPARE RÓTULO;
    RÓTULO months_ahead = 'Meses à Frente'
          stepar        = 'Previsão STEPAR'
          winters       = 'Previsão Winters'
          seasonal_gap  = 'Winters - STEPAR';
    FORMATO stepar winters seasonal_gap 8.0;
    TÍTULO 'STEPAR vs Holt-Winters: Comparação de Previsão de 12 Meses';
EXECUTAR;


                               STEPAR vs Holt-Winters: Comparação de Previsão de 12 Meses                               

  Obs   Meses à Frente   Previsão STEPAR   Previsão Winters  Winters - STEPAR
    1                1              2619               2783               164
    2                2              2537               2897               361
    3                3              2384               2805               421
    4                4              2214               2664               450
    5                5              2089               2629               540
    6                6              2010               2548               539
    7                7              1982               2392               410
    8                8              1995               2240               245
    9                9              2031               2197               166
   10               10              2075               2354               280
   11               


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Etapa 5 — Prever cada linha de negócio de uma vez (processamento BY)

Execuções de reservas reais cobrem vários produtos. Com os dados ordenados por `line_of_business`, uma instrução `BY` faz o `PROC FORECAST` ajustar um **modelo sazonal independente para cada grupo** em uma única chamada. Aqui sintetizamos uma carteira Auto (volume base mais alto) e uma carteira Casa (base mais baixa) e prevemos ambas seis meses à frente. `OUTALL` grava o conjunto completo de linhas — o histórico `ACTUAL` e o horizonte `FORECAST` — junto com as colunas de limite para cada grupo, e mantemos apenas as linhas `FORECAST` para a tabela abaixo.


In [5]:
DADOS multi_book;
    CHAMAR streaminit(20240531);
    COMPRIMENTO line_of_business $6;
    FAZER lob = 1 ATÉ 2;
        SE lob = 1 ENTÃO line_of_business = 'Auto';
        SENÃO            line_of_business = 'Casa';
        FAZER i = 0 ATÉ 47;
            date = intnx('month', '01JAN2021'd, i);
            FORMATO date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi IN (12, 1, 2))
                + rand('normal', 0, 70));
            SAÍDA;
        FIM;
    FIM;
    MANTER line_of_business date claim_count;
EXECUTAR;

PROCEDIMENTO ORDENAR DADOS=multi_book;
    POR line_of_business date;
EXECUTAR;

PROCEDIMENTO forecast DADOS=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    POR line_of_business;
    id date interval=month;
    VARIÁVEL claim_count;
EXECUTAR;

PROCEDIMENTO IMPRIMIR DADOS=fc_by(obs=12) RÓTULO;
    RÓTULO line_of_business = 'Linha de Negócio'
          date             = 'Mês'
          claim_count      = 'Sinistros'
          _type_           = 'Tipo'
          l95_claim_count  = 'Limite Inf. 95%'
          u95_claim_count  = 'Limite Sup. 95%';
    ONDE _type_ = 'FORECAST';
    TÍTULO 'Previsões de 6 Meses por Linha de Negócio';
EXECUTAR;


                               STEPAR vs Holt-Winters: Comparação de Previsão de 12 Meses                               


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Casa

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                       Previsões de 6 Meses por Linha de Negócio                                        

  Obs   Linha de Negócio    Mês      Tipo    Sinistros  Limite Inf. 95%  Limite Sup. 95%  RESIDUAL_CLAIM_COUNT
    1  Auto               23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Auto               23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Auto               23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Auto               2383


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Auto
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Casa
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretando os Resultados

**O que os modelos dizem à equipe de reservas:**

- **STEPAR** reproduz a tendência de alta e o momento de curto prazo, mas sua previsão de 12 meses decai suavemente de cerca de 2.620 (etapa 1) para cerca de 1.980 no meio do horizonte antes de voltar para cerca de 2.140 — não reproduz um pico agudo de inverno, porque trata a sazonalidade apenas por meio de defasagens autorregressivas. É uma linha de base rápida razoável.
- **WINTERS** com `SEASONS=12` carrega o ritmo anual diretamente por meio de seus fatores sazonais multiplicativos: sua previsão é mais alta no início do horizonte (cerca de 2.780 na etapa 1, cerca de 2.900 na etapa 2), suaviza para um vale perto da etapa 9 (cerca de 2.200), e sobe novamente até a etapa 12 (cerca de 2.800). Em relação à linha de base STEPAR, fica **150-660 sinistros mais alto** durante a maior parte do horizonte (veja a coluna `Winters - STEPAR` na Etapa 4) — essa diferença é o sinal sazonal que o modelo autorregressivo deixa de fora.
- A **banda de confiança de 95%** (colunas `L95_*`/`U95_*`, controladas por `ALPHA=`) se alarga à medida que o horizonte se estende — para o modelo WINTERS, de uma largura de aproximadamente 410 sinistros na etapa 1 para cerca de 1.420 na etapa 12 — o sinal honesto de que as estimativas de horizonte distante carregam mais incerteza do que as de curto prazo. Os reservistas devem reservar capital contra o limite superior, não apenas a previsão pontual.
- O **processamento BY** produz as previsões de Auto e Casa em uma única passagem, cada uma com seu próprio ajuste sazonal. A carteira Auto prevê na faixa de aproximadamente 2.510-2.600, enquanto a carteira Casa fica perto de 1.870-2.030, de modo que cada linha mantém seu próprio nível e forma sazonal — o padrão que a equipe estenderia a cada produto na carteira.

**Conclusão:** para uma série de contagem de sinistros com um ciclo anual claro, `METHOD=WINTERS SEASONS=12` é a escolha idiomática; a linha de base STEPAR é uma boa verificação de sanidade, e `OUTFULL`/`OUTLIMIT` mais uma comparação de modelo etapa a etapa permitem que o atuário dimensione a reserva prospectiva com sua banda de incerteza.

> **Nota de fidelidade do motor.** Este notebook documenta duas limitações atuais do PROC FORECAST do Jenner (teste de lacuna `400979`): o `ID` do horizonte de previsão é avançado em uma unidade por etapa em vez de por `INTERVAL=MONTH`, então as datas de previsão impressas não são os meses de calendário de 2025 pretendidos (revisamos o horizonte por etapa em vez disso); e `OUTRESID`/`OUTALL` ainda não preenchem as linhas `_TYPE_='RESIDUAL'` de um passo à frente, então os diagnósticos de resíduos são substituídos por uma comparação direta de dois modelos. As próprias previsões pontuais e os limites de confiança são produzidos e são o que a narrativa acima tem como base.
